# Script Overview
This Python script processes a set of text files to generate word count statistics for each file. It performs the following tasks:



*   Reads text files from a specified folder.
*   Preprocesses the text by converting it to lowercase, removing punctuation, numbers, and short words.
*   Counts the frequency of each word.
*   Saves the word counts as CSV files in a specified output folder.

# Dependencies
The script requires the following libraries and tools:

*   pdfminer.six (installed via !pip install pdfminer.six)
*   google.colab (for Google Drive integration)
*   Standard Python libraries: os, re, collections.Counter, pandas


In [ ]:
!pip install pdfminer.six
import os
from pdfminer.high_level import extract_text
import pandas as pd
from collections import Counter
import re

# Setup
Mount Google Drive to access files.

In [ ]:
from google.colab import drive
drive.mount('[REDACTED]')

# One word counter


Define paths for the input folder containing text files and the output folder for saving CSVs.

In [ ]:
input_folder = "[REDACTED]"  # Replace with your input folder path
output_folder = "[REDACTED]"  # Replace with your output folder path

Remove all existing files from the output folder.

In [ ]:
for filename in os.listdir(output_folder):
    file_path = os.path.join(output_folder, filename)
    try:
        if os.path.isfile(file_path) or os.path.islink(file_path):
            os.unlink(file_path)  # Remove file
    except Exception as e:
        print(f'Failed to delete {file_path}. Reason: {e}')

List all text files in the input folder.

In [ ]:
text_files = [f for f in os.listdir(input_folder) if f.endswith('.txt')]

Process each file.

In [ ]:
for text_file in text_files:
    file_path = os.path.join(input_folder, text_file)

    # Read the text file
    with open(file_path, 'r', encoding='utf-8') as f:
        text = f.read()

    # Convert text to lowercase, remove punctuation, and split words
    text = re.sub(r'[^\w\s]', '', text.lower())  # Remove punctuation and convert to lowercase
    words = text.split()  # Split text into words

    # Remove numbers and one or two letter words
    filtered_words = [word for word in words if len(word) > 2 and word.isalpha()]

    # Create a word counter
    word_count = Counter(filtered_words)

    # Convert the counter to a DataFrame and sort by 'Count' in descending order
    df = pd.DataFrame(word_count.items(), columns=['Word', 'Count']).sort_values(by='Count', ascending=False)

    # Save the DataFrame to a CSV file in the output folder
    csv_file_path = os.path.join(output_folder, f'{os.path.splitext(text_file)[0]}_word_count.csv')
    df.to_csv(csv_file_path, index=False)

    print(f'Word count CSV saved for {text_file} at {csv_file_path}')

# Two word counter

In [ ]:
# Define the folder containing text files and the output folder for CSVs
input_folder = "[REDACTED]"  # Replace with your folder path
output_folder = "[REDACTED]"  # Replace with your output folder path

# Clear the contents of the output folder without deleting the folder itself
for filename in os.listdir(output_folder):
    file_path = os.path.join(output_folder, filename)
    try:
        if os.path.isfile(file_path) or os.path.islink(file_path):
            os.unlink(file_path)  # Remove file
    except Exception as e:
        print(f'Failed to delete {file_path}. Reason: {e}')

# Define an extended list of common stop words
stop_words = {
    "the", "and", "in", "to", "of", "a", "is", "that", "for", "it", "on", "with", "as", "was", "at", "by", "an",
    "be", "this", "which", "or", "from", "but", "not", "are", "have", "has", "we", "you", "he", "she", "they",
    "them", "his", "her", "their", "its", "my", "me", "i", "our", "us", "who", "whom", "what", "when", "where",
    "why", "how", "can", "will", "would", "could", "should", "all", "any", "each", "every", "some", "no", "do",
    "does", "did", "so", "than", "then", "up", "out", "if", "about", "into", "like", "just", "over", "after",
    "such", "even", "more", "also", "other", "many", "these", "those", "now", "only"
}

# List all text files in the input folder
text_files = [f for f in os.listdir(input_folder) if f.endswith('.txt')]

# Process each text file
for text_file in text_files:
    file_path = os.path.join(input_folder, text_file)

    # Read the text file
    with open(file_path, 'r', encoding='utf-8') as f:
        text = f.read()

    # Convert text to lowercase, remove punctuation, and split words
    text = re.sub(r'[^\w\s]', '', text.lower())  # Remove punctuation and convert to lowercase
    words = text.split()  # Split text into words

    # Remove numbers, one or two letter words, and stop words
    filtered_words = [word for word in words if len(word) > 2 and word.isalpha() and word not in stop_words]

    # Generate word pairs (bigrams)
    bigrams = [(filtered_words[i], filtered_words[i+1]) for i in range(len(filtered_words) - 1)]

    # Create a bigram counter
    bigram_count = Counter(bigrams)

    # Convert the counter to a DataFrame and sort by 'Count' in descending order
    df = pd.DataFrame(bigram_count.items(), columns=['Bigram', 'Count']).sort_values(by='Count', ascending=False)

    # Save the DataFrame to a CSV file in the output folder
    csv_file_path = os.path.join(output_folder, f'{os.path.splitext(text_file)[0]}_bigram_count.csv')
    df.to_csv(csv_file_path, index=False)

    print(f'Bigram count CSV saved for {text_file} at {csv_file_path}')

Document and word mapping



In [ ]:
# Define the root folder containing subfolders with CSV files
input_folder = "[REDACTED]"  # Folder containing subfolders with CSV files
output_folder = "[REDACTED]"  # Folder to save the output CSV file

# Ensure the output folder exists
os.makedirs(output_folder, exist_ok=True)

# Clear the output folder of any existing files
for filename in os.listdir(output_folder):
    file_path = os.path.join(output_folder, filename)
    try:
        if os.path.isfile(file_path) or os.path.islink(file_path):
            os.unlink(file_path)  # Remove the file
    except Exception as e:
        print(f'Failed to delete {file_path}. Reason: {e}')

# Path to save the output CSV file
output_csv_path = os.path.join(output_folder, '[REDACTED].csv')

# Initialize a list to collect document-word/bigram data
document_word_data = []

# Walk through each folder and file in the input folder
for subdir, _, files in os.walk(input_folder):
    for file in files:
        if file.endswith('.csv'):
            # Get the path of the current CSV file
            csv_path = os.path.join(subdir, file)

            # Read the CSV file
            df = pd.read_csv(csv_path)

            # Determine if the CSV has a "Word" or "Bigram" column
            if 'Word' in df.columns:
                word_column = 'Word'
            elif 'Bigram' in df.columns:
                word_column = 'Bigram'
            else:
                print(f"Warning: CSV file {file} does not contain 'Word' or 'Bigram' columns. Skipping this file.")
                continue

            # Get the document name from the file name
            document_name = os.path.splitext(file)[0]

            # Append each unique word or bigram in the document to document_word_data
            for term in df[word_column].unique():
                # If it's a bigram stored as a tuple, convert it to a string with a space
                if isinstance(term, str):
                    term = term.replace("(", "").replace(")", "").replace(",", "").replace("'", "").strip()
                document_word_data.append({
                    'Document Name': document_name,
                    'Word or Bigram': term
                })

# Convert document_word_data to a DataFrame
output_df = pd.DataFrame(document_word_data)

# Save the DataFrame to a CSV file in the specified output folder
output_df.to_csv(output_csv_path, index=False)

print(f'Document-word or bigram mapping CSV saved at {output_csv_path}')

File matching (pdf and text files)

In [ ]:
# Define the folders
pdf_folder = "[REDACTED]"  # Folder containing corresponding PDF files
text_folder = "[REDACTED]'s transcripts ([REDACTED] videos)"  # Folder containing text files
output_csv_folder = "[REDACTED]"  # Path to the output CSV file

# Define Google Drive folder IDs (replace with your actual folder IDs)
pdf_folder_id = "[REDACTED]"  # Replace with your Google Drive PDF folder ID
text_folder_id = "[REDACTED]"  # Replace with your Google Drive text folder ID

# Ensure the output folder exists
os.makedirs(output_csv_folder, exist_ok=True)

# Path to save the output CSV file
output_csv_path = os.path.join(output_csv_folder, 'matched_files_with_links.csv')

# Initialize a list to collect matched data
matched_data = []

# Process each text file
for text_file in os.listdir(text_folder):
    if text_file.endswith('.txt'):
        # Define corresponding PDF filename
        pdf_file = text_file.replace('.txt', '.pdf')

        # Check if the corresponding PDF file exists in the PDF folder
        pdf_path = os.path.join(pdf_folder, pdf_file)
        if os.path.isfile(pdf_path):
            # Construct Google Drive links for the PDF file and text file
            pdf_link = f'[REDACTED]'
            text_link = f'[REDACTED]'

            # Add the matched text and PDF filenames, and their links, to matched_data
            matched_data.append({
                'PDF Filename': pdf_file,
                'PDF Link': pdf_link,
                'Text Filename': text_file,
                'Text Link': text_link
            })
        else:
            print(f'Warning: PDF file {pdf_file} not found for {text_file}. Skipping this file.')

# Convert matched_data to a DataFrame
df = pd.DataFrame(matched_data)

# Save the DataFrame to a CSV file in the specified output folder
df.to_csv(output_csv_path, index=False)

print(f'Matched files with links CSV saved at {output_csv_path}')